## Reading 38-2 - The Kalman Filter

By the end of this reading, you will learn:
<ol>
    <li>The idea behind the Kalman filter: blend a prediction and a measurement, each weighted by how much you trust it</li>
    <li>The three steps of each cycle: predict, compute the gain, update</li>
    <li>How the setting Q controls the trade-off between noise and lag</li>
    <li>How the same filter detects faulty measurements, and how it connects to the leaky bucket</li>
</ol>

### Two Sources of Information

At every time step, a spacecraft has two imperfect pieces of information about a temperature:
<ul>
    <li>A <b>prediction</b>, based on what it believed a moment ago. Temperatures change slowly, so the last estimate is a good guess for the next one.</li>
    <li>A <b>measurement</b> from the sensor, which is current but noisy.</li>
</ul>

The Kalman filter, developed by Rudolf Kalman in 1960 and used to navigate the Apollo spacecraft to the Moon, combines them. Ama's slide summarizes it in one line: a model of a system and a measurement, <b>each weighted by how much you trust it</b>.

The filter keeps two numbers:
<ul>
    <li><code>x</code>, the current <b>estimate</b> of the temperature</li>
    <li><code>P</code>, the <b>variance</b> of that estimate: how unsure the filter is about it</li>
</ul>

And it uses two settings:
<ul>
    <li><code>R</code>, the <b>measurement noise</b>: the sensor's variance, 0.528, from <code>noise_stats</code> in Reading 38-1</li>
    <li><code>Q</code>, the <b>process noise</b>: how much the true temperature can change between samples</li>
</ul>

### One Cycle - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec38/kalman.c">kalman.c</a>

Ama's slide shows each cycle as three steps. For a single temperature, each step is one or two lines of C:

    double kalman_step(kalman1d *kf, double z)
    {
    	/* Predict: the temperature stays the same, but we grow less sure */
    	double x_pred = kf->x;
    	double P_pred = kf->P + kf->Q;

    	/* Gain: how much to trust the measurement over the prediction */
    	double K = P_pred / (P_pred + kf->R);

    	/* Update: move toward the measurement by K, and become more sure */
    	kf->x = x_pred + K * (z - x_pred);
    	kf->P = (1.0 - K) * P_pred;
    	return K;
    }

The <b>gain</b> <code>K</code> is always between 0 and 1, and it is the key to the whole filter:
<ul>
    <li>If the prediction is very uncertain (<code>P_pred</code> large compared to <code>R</code>), <code>K</code> is close to 1: trust the sensor.</li>
    <li>If the sensor is very noisy (<code>R</code> large compared to <code>P_pred</code>), <code>K</code> is close to 0: trust the prediction.</li>
</ul>

    $ make kalman
    The first steps, with Q = 0.1:
    sample  measured  estimate  gain K variance P
         1     20.59     20.51   0.543     0.287
         2     18.65     19.72   0.423     0.223
         3     20.31     19.95   0.380     0.201
         4     19.35     19.73   0.363     0.192
         5     19.85     19.77   0.356     0.188
         6     19.86     19.80   0.353     0.186

At first, the filter knows little, so it trusts the sensor heavily (<code>K</code> = 0.543). As measurements accumulate, <code>P</code> shrinks and <code>K</code> settles to a steady value of about 0.35. Notice how little the estimate moved at sample 2, when the sensor read a noisy 18.65 C.

Compare the update line to exponential smoothing in Reading 38-1:

    ema = ema + ALPHA * (meas - ema);

They are the same equation, with <code>K</code> in place of <code>ALPHA</code>. Once <code>K</code> settles, this one-dimensional Kalman filter <i>is</i> exponential smoothing. The difference is that the Kalman filter <b>calculates</b> the right weight from the measured noise instead of guessing it, and it knows its own uncertainty, <code>P</code>, at every step. That second difference is what makes it a fault detector.

### Tuning Q

<code>R</code> comes from calibration, but <code>Q</code> is a judgment about how fast the true temperature can change. Here is the whole mission with four values:

    Tuning Q (R = 0.528 from calibration):
           Q  final K  RMS error lag after step
       0.001    0.043      1.352          never
       0.010    0.128      0.585             11
       0.100    0.351      0.534              3
       1.000    0.724      0.823              0

<ul>
    <li><b>Q too small:</b> the filter believes the temperature barely changes, so it trusts its prediction and ignores the sensor. It never caught up after the step at sample 36. This is the lag on Ama's slide, where the estimate trails the true value after the jump.</li>
    <li><b>Q too large:</b> the filter believes the temperature can change a lot, so it trusts every noisy reading. No lag, but noisy output.</li>
    <li><b>Q = 0.1</b> balances the two for this mission: lowest error, caught up 3 samples after the step.</li>
</ul>

This is the same noise-versus-lag trade-off as Reading 38-1, but now it is controlled by one number with a physical meaning.

### The Filter as a Fault Detector - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec38/innovation.c">innovation.c</a>

Before each update, the filter predicts the next reading. The difference between what the sensor actually reports and that prediction is called the <b>innovation</b>. The filter also knows how big the innovation should typically be: its variance is <code>S = P_pred + R</code>. Dividing one by the square root of the other gives a number in standard deviations:

    double innovation = z - x;
    double S = P_pred + R_MEASURED;
    double surprise = innovation / sqrt(S);

From Reading 38-1, a reading more than 3 standard deviations from the prediction should be rare. So the program applies a <b>gate</b>: a measurement with <code>|surprise| > 3</code> is rejected and not allowed to move the estimate. And it uses the <b>leaky bucket</b> from Lecture 37 to decide whether the rejections add up to a real fault:

    $ make innovation
    ./innovation mission.csv
    sample 26: measured  24.63, predicted  21.51,  3.5 std devs: REJECTED (bucket 2)
    sample 53: measured  29.98, predicted  25.70,  4.7 std devs: REJECTED (bucket 2)
    Mission complete: no sensor fault confirmed. Final estimate 26.67 C (true 25.84 C)

    ./innovation mission.csv bias
    sample 26: measured  24.63, predicted  21.51,  3.5 std devs: REJECTED (bucket 2)
    sample 46: measured  29.58, predicted  25.69,  4.3 std devs: REJECTED (bucket 2)
    sample 47: measured  29.27, predicted  25.69,  3.7 std devs: REJECTED (bucket 4)
              SENSOR FAULT confirmed: switch to backup sensor

In the first run, two isolated readings were rejected, one a natural noise spike and one a glitch. Neither repeated, so the bucket drained and no fault was declared. The estimate was never pulled off course by either one.

In the second run, the sensor develops a +3.0 C bias at sample 45. Every reading after that is surprising, the rejections pile up, and the fault is confirmed within two samples. The software can now switch to a backup sensor, the next step on the escalation ladder from Reading 37-2.

This is what Ama means by "the same filter as a noise model and as a fault detector." Everything you have built in Lectures 36, 37, and 38, time stamps, voting, leaky buckets, and filtering, works together in real fault protection.

### <font color = "red">Class Introduction Question #4 - In <code>kalman_step</code>, what happens to the gain <code>K</code> when the sensor is very noisy? When the prediction is very uncertain? Why did <code>K</code> start near 0.5 and settle near 0.35?</font>

### <font color = "red">Class Introduction Question #5 - Explain why <code>Q</code> = 0.001 never caught up after the step, and why <code>Q</code> = 1.0 had a larger RMS error than <code>Q</code> = 0.1.</font>

### <font color = "red">Class Introduction Question #6 - What is the innovation? In <code>innovation.c</code>, why was the reading at sample 53 rejected but not treated as a sensor fault, while the readings after sample 45 in the second run were?</font>

### In class, we will write a Kalman filter for a spacecraft temperature sensor, test it to a tolerance, and use it to catch a failing sensor.